In [1]:
import h5pyd
import h5py
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import haversine
from haversine import haversine, Unit
import numpy as np
from datetime import datetime

In [2]:
sites = pd.read_csv('C:/Users/sarah/Documents/GitHub/ORKA_StatusReport_FollowupPaper/StatusReport_FollowupPaper_Data/MasterSiteList_FollowupPaper.csv')
sites = sites.iloc[0:15,0:8]

In [17]:
#Combine the two datasets together. Realized that the initial ROMS dataframe was cropped just a little bit too far west and we needed to add back
#in a bit of data from further east in the larger dataset. Ended up just combining the two portions of raw data together in this step instead of 
#making a new clean one for the sake of efficiency and dealing with these huge freaking datasets. Not pretty, but it worked.
#Combine the two datasets together
#temp is 235,526,278 rows long
temp = pd.read_csv("D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles/surface_temp_all.csv")
temp = temp.dropna()
#temp_east is 10,733,967 rows long
temp_east = pd.read_csv("D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles/surface_temp_east_all.csv")
temp = pd.concat([temp,temp_east])
print("Number of time steps is", temp.ocean_time.nunique())
print("Lat/lon combos is", len(temp[['lon_rho','lat_rho']].drop_duplicates()))
#drop 2024 values and reset index
temp.ocean_time = pd.to_datetime(temp.ocean_time)
temp = temp.loc[temp.ocean_time < "2023-12-31"].reset_index()

Number of time steps is 4383
Lat/lon combos is 48278


In [52]:
#%%time
#Ok here I'm finding which ROMS grid cell is closest to each site 
#then pulling out the ROMS data for each nearest grid cell and putting it into a dataframe called 'temp_forsites'
lat = sites.SiteLatitude[0]
lon = sites.SiteLongitude[0]
site = (lat, lon)
site_rounded = (round(lat,2), round(lon, 2))
temp_sub = temp.loc[(temp.lat_rho < site[0] +0.01) & (temp.lat_rho > site[0]-0.01)]
def hav_dist(row):
    site2 = (row["lat_rho"],row["lon_rho"])
    return haversine(site,site2)
dist = temp_sub.apply(hav_dist, axis = 1)
index1 = dist[dist == dist.min()].index
temp_forsites = temp.iloc[index1.values[:],]

for i in range(1,len(sites)): 
    #print(i, "has the latitude", sites.SiteLatitude[i])
    site = (sites.SiteLatitude[i], sites.SiteLongitude[i])
    temp_sub = temp.loc[(temp.lat_rho < site[0] +0.01) & (temp.lat_rho > site[0]-0.01)]
    dist = temp_sub.apply(hav_dist, axis = 1)
    index1 = dist[dist == dist.min()].index
    closest = temp.iloc[index1.values[:],]
    print(i)
    print(closest.iloc[1,6])
    temp_forsites = pd.concat([temp_forsites,closest])
    print(i," is finished, hooray!")
temp_forsites.ocean_time = pd.to_datetime(temp_forsites.ocean_time)

1
-123.98801621298482
1  is finished, hooray!
2
-124.07740957398244
2  is finished, hooray!
3
-124.07037017608252
3  is finished, hooray!
4
-124.37491972563382
4  is finished, hooray!
5
-124.4081185068446
5  is finished, hooray!
6
-124.3997693423036
6  is finished, hooray!
7
-124.58225216739925
7  is finished, hooray!
8
-124.5913269255099
8  is finished, hooray!
9
-124.51972726039511
9  is finished, hooray!
10
-124.4674996371059
10  is finished, hooray!
11
-124.44184846960437
11  is finished, hooray!
12
-124.4674996371059
12  is finished, hooray!
13
-124.31807492787684
13  is finished, hooray!
14
-124.2941921197234
14  is finished, hooray!


In [53]:
#Check which grid cells from ROMs model are pulled out to correspond with which sites
temp_forsites[["lat_rho","lon_rho"]].drop_duplicates()

,lat_rho,lon_rho
178916759,45.335118,-123.981283
178916651,45.213317,-123.988016
21078,44.783831,-124.077410
20654,44.749722,-124.070370
6739,43.342748,-124.374920
6606,43.320523,-124.408119
6479,43.298217,-124.399769
4073,42.834800,-124.582252
3882,42.786595,-124.591327
3692,42.738041,-124.519727


In [54]:
#90th percentile of temperature values at each site
temp90 = temp_forsites.groupby("lat_rho").surf_temp.quantile(q = 0.9).reset_index()
temp90

#Adding this new temp metric for each site into a dataframe that shows multiple metrics for each site
temp90_reordered = temp90.iloc[::-1].reset_index()
sites["P90_temp"] = temp90_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P90_temp,Temp_Over15,P90_temp_growing,P90_temp_winter
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,13.293250,1.505818,13.726345,12.668628
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,13.396740,1.711157,13.843090,12.542642
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,12.860991,0.634398,12.861338,12.743552
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,12.702344,0.587406,12.647614,12.638301
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,13.111904,1.221805,13.169181,12.933340
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,13.115160,1.174812,13.164645,12.923909
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,12.887273,0.704887,12.697789,12.900149
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,12.865562,0.892857,12.479222,13.027904
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,12.913808,1.057331,12.469853,13.052721
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,12.868502,0.704887,12.295246,13.055692


In [55]:
#Create a function to calculate the percentage of temp above 15 degrees C
def time_above_15(data):
    return len(data.loc[data.surf_temp > 15])/len(data)*100
tempover15 = temp_forsites.groupby("lat_rho").apply(time_above_15).reset_index()
#Adding this new temp metric for each site into a dataframe that shows multiple metrics for each site
tempover15_reordered = tempover15.iloc[::-1].reset_index()
sites["Temp_Over15"] = tempover15_reordered.iloc[:,2]
sites

C:\Users\sarah\AppData\Local\Temp\ipykernel_28040\3369998334.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tempover15 = temp_forsites.groupby("lat_rho").apply(time_above_15).reset_index()


,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P90_temp,Temp_Over15,P90_temp_growing,P90_temp_winter
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,13.293250,1.543825,13.726345,12.668628
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,13.396740,1.817729,13.843090,12.542642
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,12.860991,0.617125,12.861338,12.743552
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,12.702344,0.565698,12.647614,12.638301
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,13.111904,1.157110,13.169181,12.933340
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,13.115160,1.105683,13.164645,12.923909
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,12.887273,0.617125,12.697789,12.900149
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,12.865562,0.822834,12.479222,13.027904
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,12.913808,1.079969,12.469853,13.052721
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,12.868502,0.771407,12.295246,13.055692


In [56]:
#growing season 90th percentile of temperature values at each site
temp_growing = temp_forsites.loc[(temp_forsites.ocean_time.dt.month>=4) & (temp_forsites.ocean_time.dt.month<=9)]
temp90_growing = temp_growing.groupby("lat_rho").surf_temp.quantile(q = 0.9).reset_index()
temp90_growing

#Adding this new temp metric for each site into a dataframe that shows multiple metrics for each site
temp90_growing_reordered = temp90_growing.iloc[::-1].reset_index()
sites["P90_temp_growing"] = temp90_growing_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P90_temp,Temp_Over15,P90_temp_growing,P90_temp_winter
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,13.293250,1.543825,13.777001,12.668628
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,13.396740,1.817729,13.877231,12.542642
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,12.860991,0.617125,12.912010,12.743552
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,12.702344,0.565698,12.688675,12.638301
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,13.111904,1.157110,13.169568,12.933340
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,13.115160,1.105683,13.165854,12.923909
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,12.887273,0.617125,12.746955,12.900149
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,12.865562,0.822834,12.511692,13.027904
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,12.913808,1.079969,12.520105,13.052721
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,12.868502,0.771407,12.371046,13.055692


In [57]:
#winter season 90th percentile of temperature values at each site
temp_winter = temp_forsites.loc[(temp_forsites.ocean_time.dt.month<=3) | (temp_forsites.ocean_time.dt.month>=10)]
temp90_winter = temp_winter.groupby("lat_rho").surf_temp.quantile(q = 0.9).reset_index()
temp90_winter

#Adding this new temp metric for each site into a dataframe that shows multiple metrics for each site
temp90_winter_reordered = temp90_winter.iloc[::-1].reset_index()
sites["P90_temp_winter"] = temp90_winter_reordered.iloc[:,2]
sites

,Site,SiteCode,Area,AreaCode,Port,SiteLatitude,SiteLongitude,Number2023Trans_Estiamted,P90_temp,Temp_Over15,P90_temp_growing,P90_temp_winter
0,Cape Lookout,LOOK,Cape Lookout,LKOUT,Pacific City,45.336730,-123.980000,0.0,13.293250,1.543825,13.777001,12.760380
1,Pacific City,PACC,Cape Lookout,LKOUT,Pacific City,45.212470,-123.985186,12.0,13.396740,1.817729,13.877231,12.629706
2,Cape Foulweather,FOUL,Cape Foulweather,FOULW,Depoe Bay,44.784468,-124.074200,44.0,12.860991,0.617125,12.912010,12.800043
3,Otter Rock,OTRK,Cape Foulweather,FOULW,Depoe Bay,44.746630,-124.072591,52.0,12.702344,0.565698,12.688675,12.747765
4,Gregory Point,GREG,Cape Arago,ARAGO,Coos Bay,43.339313,-124.378642,10.0,13.111904,1.157110,13.169568,13.039712
5,Simpson Reef,SIMP,Cape Arago,ARAGO,Coos Bay,43.315740,-124.404422,16.0,13.115160,1.105683,13.165854,13.027307
6,Drake Point,DRAK,Cape Arago,ARAGO,Coos Bay,43.300839,-124.400720,14.0,12.887273,0.617125,12.746955,12.980334
7,Blanco Reef,BLCO,Cape Blanco,BLANC,Port Orford,42.831791,-124.581814,16.0,12.865562,0.822834,12.511692,13.119866
8,Orford Reef,ORRF,Cape Blanco,BLANC,Port Orford,42.788491,-124.595368,26.0,12.913808,1.079969,12.520105,13.155018
9,Port Orford Heads,ORFH,Cape Blanco,BLANC,Port Orford,42.735759,-124.507747,26.0,12.868502,0.771407,12.371046,13.222920


In [59]:
sites.to_csv('C:/Users/sarah/Documents/GitHub/ORKA_StatusReport_FollowupPaper/StatusReport_FollowupPaper_Data/MasterSites_WithTempMetrics_20132023.csv')